In [2]:
from langchain_core.documents import Document

In [3]:
documents = [
    Document(page_content=" Batman is a superhero who fights crime in Gotham City.", metadata={"source": "https://en.wikipedia.org/wiki/Batman"}),
    Document(page_content=" Batman's real name is Bruce Wayne.", metadata={"source": "https://en.wikipedia.org/wiki/Batman"}),
    Document(page_content=" Batman's sidekick is Robin.", metadata={"source": "https://en.wikipedia.org/wiki/Batman"}),
    Document(page_content=" Batman's arch-nemesis is the Joker.", metadata={"source": "https://en.wikipedia.org/wiki/Batman"}),
    Document(page_content=" Batman's love interest is Catwoman.", metadata={"source": "https://en.wikipedia.org/wiki/Batman"}),
    Document(page_content=" Batman's parents were murdered when he was a child.", metadata={"source": "https://en.wikipedia.org/wiki/Batman"}),
    Document(page_content=" Batman's Batmobile is a high-tech vehicle equipped with various gadgets.", metadata={"source": "https://en.wikipedia.org/wiki/Batmobile"})]

In [4]:
documents[2].page_content

" Batman's sidekick is Robin."

In [53]:
import os
from dotenv import load_dotenv
load_dotenv()
from langchain_groq import ChatGroq

groq_api_key = os.getenv("GROQ_API_KEY")

os.environ["HUGGINGFACEHUB_API_TOKEN"]= os.getenv("HUGGINGFACEHUB_API_TOKEN")

llm= ChatGroq(groq_api_key=groq_api_key, model = "llama-3.1-8b-instant")

In [54]:
from langchain_huggingface import HuggingFaceEmbeddings
embedding = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2") 

## Vector store

In [55]:
from langchain_chroma import Chroma 
vectorsotre = Chroma.from_documents(documents, embedding) 
vectorsotre

In [56]:
vectorsotre.similarity_search("What is Batman's real name?", k=1)

[Document(id='0bbb7e82-bf4e-4ef9-848d-6021d358f77b', metadata={'source': 'https://en.wikipedia.org/wiki/Batman'}, page_content=" Batman's real name is Bruce Wayne.")]

## Async query

In [57]:
await vectorsotre.asimilarity_search("What is Batman's real name?", k=1)

[Document(id='0bbb7e82-bf4e-4ef9-848d-6021d358f77b', metadata={'source': 'https://en.wikipedia.org/wiki/Batman'}, page_content=" Batman's real name is Bruce Wayne.")]

In [58]:
vectorsotre.similarity_search_with_score("What is Batman's real name?", k=1)

[(Document(id='0bbb7e82-bf4e-4ef9-848d-6021d358f77b', metadata={'source': 'https://en.wikipedia.org/wiki/Batman'}, page_content=" Batman's real name is Bruce Wayne."),
  0.26931867003440857)]

## Retrievers 

In [59]:
from typing import List

from langchain_core.documents import Document
from langchain_core.runnables import RunnableLambda

retriever = RunnableLambda(vectorsotre.similarity_search).bind(k=1)
retriever_output= retriever.batch(["What is Catwoman's real name?"])

In [60]:
retriever_output

[[Document(id='9d6cae1d-ddb9-40b0-aca9-eb3cbd05c7bc', metadata={'source': 'https://en.wikipedia.org/wiki/Batman'}, page_content=" Batman's love interest is Catwoman.")]]

### Implemnting vectcorstores as as_retriever 
##### This will generate a retriever, specifically a VectorStoreRretriever

In [61]:
vectorsotre.as_retriever (search_type="similarity", search_kwargs={"k": 1})
retriever.batch(["What is Joker's real name?"])

[[Document(id='cbe63257-bca7-43f4-b9a9-3eb95ae2a667', metadata={'source': 'https://en.wikipedia.org/wiki/Batman'}, page_content=" Batman's arch-nemesis is the Joker.")]]

In [62]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough

message = """ Answer this question using the provided context only. 
Question: {question}
Context: {context}"""


prompt = ChatPromptTemplate.from_messages([("system", "You are a helpful assistant."), ("user", message)])

rag_chain = {"context": retriever, "question": RunnablePassthrough()} | prompt | llm
rag_chain.invoke("Tell me about Batman's love interest.")


AIMessage(content="Batman's love interest is Catwoman.", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 9, 'prompt_tokens': 119, 'total_tokens': 128, 'completion_time': 0.011714586, 'completion_tokens_details': None, 'prompt_time': 0.01016015, 'prompt_tokens_details': None, 'queue_time': 0.006469644, 'total_time': 0.021874736}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_d317489708', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019cf75f-fe76-7342-b052-2bde47dd7f36-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 119, 'output_tokens': 9, 'total_tokens': 128})